# Task — Gold KPIs (rebuilt from silver)

**The gold layer here = business KPIs.** One single SQL statement — `CREATE OR REPLACE TABLE` — rebuilds the KPI table fresh from silver on every run. That is exactly why CREATE OR REPLACE is used: it drops and rebuilds the table, so the KPIs are **always correct and never duplicated**, no matter how often the job runs.

KPIs per category: total orders, unique customers, total revenue, average order value.

In [0]:
# CELL 1 — Parameters
dbutils.widgets.text("catalog", "demo1", "Catalog that holds all demo tables")
dbutils.widgets.text("job_id", "interactive", "Job ID (passed by the job)")
dbutils.widgets.text("run_id", "interactive", "Run ID (passed by the job)")

catalog = dbutils.widgets.get("catalog")
job_id = dbutils.widgets.get("job_id")
run_id = dbutils.widgets.get("run_id")
print(f"catalog : {catalog} | job_id / run_id: {job_id} / {run_id}")

In [0]:
# CELL 2 — the whole gold layer in ONE SQL statement (nothing else!)
# CREATE OR REPLACE = the KPI table is rebuilt fresh from silver on every run.
KPI_TABLE = f"{catalog}.gold.sales_kpi"

spark.sql(f"""
    CREATE OR REPLACE TABLE {KPI_TABLE} AS
    SELECT
        category,
        count(*)                      AS total_orders,
        count(DISTINCT customer_name)  AS unique_customers,
        round(sum(total_amount), 2)    AS total_revenue,
        round(avg(total_amount), 2)    AS avg_order_value
    FROM {catalog}.silver.sales_orders
    GROUP BY category
    ORDER BY total_revenue DESC
""")

print(f"KPIs rebuilt in {KPI_TABLE}:")
display(spark.table(KPI_TABLE))

In [0]:
# CELL 3 — audit log row (the demo's diary — with the real run_id)
from datetime import datetime

kpi_rows = spark.table(f"{catalog}.gold.sales_kpi").count()
log_df = spark.createDataFrame(
    [(datetime.now(), job_id, run_id, "gold_kpi", int(kpi_rows), "SUCCESS",
      f"rebuilt gold sales_kpi ({kpi_rows} category rows)")],
    "logged_at TIMESTAMP, job_id STRING, run_id STRING, task_name STRING, "
    "rows_processed BIGINT, status STRING, message STRING",
)
log_df.write.mode("append").saveAsTable(f"{catalog}.bronze.job_run_log")
print("Audit row written.")

## Gold KPI — recap

* One `CREATE OR REPLACE TABLE` statement rebuilt the KPIs from silver — safe to run any number of times, always current, never duplicated.
* Show this table as the final slide of the pipeline: **bronze (raw history) → silver (clean, one row per order, SCD1) → gold (business KPIs)**.